In [0]:

# ============================================================================
# NOTEBOOK DATABRICKS - MIGRATION COUCHE GOLD (PySpark + Delta)
# Objectif : Charger les fichiers CSV Gold générés localement et les convertir
#           en tables Delta managées dans Databricks pour exploitation analytique
# ============================================================================

# MAGIC %md
# MAGIC ## Phase 1 : Configuration des chemins d'accès

# Définition du volume de stockage Databricks contenant les fichiers CSV Gold
VOLUME_PATH = "/Volumes/workspace/default/eau/gold"

# Dictionnaire des fichiers CSV produits par l'étape précédente
# Chaque clé représente le nom logique de la table, la valeur le nom du fichier source
GOLD_FILES = {
    "kpis": "gold_global_kpis.csv",
    "departements": "gold_departments_pollution.csv", 
    "communes_best": "gold_communes_top10_best.csv",
    "communes_worst": "gold_communes_top10_worst.csv",
    "critical": "gold_critical_parameters.csv"
}

# ============================================================================
# Phase 2 : Vérification de l'existence des fichiers sources
# ============================================================================

print("Recherche des fichiers dans : " + VOLUME_PATH)
print("=" * 60)

files_found = 0
files_missing = 0

for key, filename in GOLD_FILES.items():
    filepath = VOLUME_PATH + "/" + filename
    try:
        if dbutils.fs.ls(filepath):
            print("OK - " + filename + " - Trouve")
            files_found = files_found + 1
    except Exception:
        print("MANQUANT - " + filename + " - Non trouve")
        files_missing = files_missing + 1

print("\nResume : " + str(files_found) + " trouves, " + str(files_missing) + " manquants")
print("=" * 60)

# ============================================================================
# Phase 3 : Chargement des fichiers CSV en DataFrames PySpark
# ============================================================================

gold_tables = {}

for key, filename in GOLD_FILES.items():
    filepath = VOLUME_PATH + "/" + filename
    try:
        print("\nChargement : " + filename)
        
        # Lecture du CSV avec détection automatique du schéma
        # Séparateur point-virgule car export standard depuis pandas
        df = spark.read.csv(filepath, header=True, inferSchema=True, sep=";")
        
        print("   OK - " + str(df.count()) + " lignes x " + str(len(df.columns)) + " colonnes")
        
        # Ajout de métadonnées de traçabilité pour l'audit du pipeline
        from pyspark.sql.functions import col, lit, current_timestamp
        df = df.withColumn("loaded_at", current_timestamp()) \
               .withColumn("source_file", lit(filename))
        
        gold_tables[key] = df
        
        # Affichage des premières colonnes pour vérification rapide
        print("   Colonnes : " + str(df.columns[:5]))
        
    except Exception as e:
        print("   ERREUR sur " + filename + " : " + str(e))
        gold_tables[key] = None

# ============================================================================
# Phase 4 : Conversion des DataFrames en tables Delta managées
# ============================================================================

print("\n" + "=" * 60)
print("Conversion en tables Delta")
print("=" * 60)

tables_created = 0

for key, df in gold_tables.items():
    if df is not None:
        table_name = "gold_" + key
        
        print("\nCreation de la table Delta : " + table_name)
        
        try:
            # Écriture en tant que table Delta managée par Databricks
            # Le mode overwrite remplace la table si elle existe déjà
            # overwriteSchema permet de mettre à jour la structure si necessaire
            df.write.format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(table_name)
            
            # Verification post-ecriture
            count = spark.table(table_name).count()
            print("   OK - Table creee : " + str(count) + " lignes")
            tables_created = tables_created + 1
            
        except Exception as e:
            print("   ERREUR - Creation table : " + str(e))

print("\nTotal tables creees : " + str(tables_created) + "/" + str(len(gold_tables)))

# ============================================================================
# Phase 5 : Validation des tables Delta avec requetes SQL
# ============================================================================

print("\n" + "=" * 60)
print("Verification des tables creees")
print("=" * 60)

# Liste des tables Gold disponibles dans le catalogue
print("\nTables Gold disponibles :")
tables = spark.sql("SHOW TABLES LIKE 'gold_*'")
display(tables)

# Requete de validation sur la table des KPIs globaux
if gold_tables.get("kpis") is not None:
    print("\nKPIs Globaux :")
    try:
        result = spark.sql("""
            SELECT 
                total_records,
                global_conformity_rate,
                departments_covered,
                communes_covered
            FROM gold_kpis
        """)
        display(result)
    except Exception as e:
        print("   Attention - Colonnes differentes de celles attendues : " + str(e))

# Requete de validation sur la table des departements
if gold_tables.get("departements") is not None:
    print("\nTop 5 Departements par niveau de pollution :")
    try:
        result = spark.sql("""
            SELECT 
                code_departement,
                pollution_score,
                alerts
            FROM gold_departements
            ORDER BY pollution_score DESC
            LIMIT 5
        """)
        display(result)
    except Exception as e:
        print("   Erreur requete departements : " + str(e))

# Requete de validation sur la table des meilleures communes
if gold_tables.get("communes_best") is not None:
    print("\nTop 3 Meilleures Communes :")
    try:
        result = spark.sql("SELECT * FROM gold_communes_best LIMIT 3")
        display(result)
    except Exception as e:
        print("   Erreur requete communes best : " + str(e))

# Requete de validation sur la table des communes les plus polluees
if gold_tables.get("communes_worst") is not None:
    print("\nTop 3 Communes les plus polluees :")
    try:
        result = spark.sql("SELECT * FROM gold_communes_worst LIMIT 3")
        display(result)
    except Exception as e:
        print("   Erreur requete communes worst : " + str(e))

# Requete de validation sur la table des parametres critiques
if gold_tables.get("critical") is not None:
    print("\nParametres Critiques :")
    try:
        result = spark.sql("""
            SELECT 
                parameter,
                count as nb_mesures,
                mean as valeur_moyenne,
                percent_exceeding as pct_hors_seuil
            FROM gold_critical
            ORDER BY percent_exceeding DESC
        """)
        display(result)
    except Exception as e:
        print("   Erreur requete parametres critiques : " + str(e))

# ============================================================================
# Phase 6 : Creation d'agregations analytiques supplementaires
# ============================================================================

print("\n" + "=" * 60)
print("Creation d'agregations complementaires")
print("=" * 60)

# Agregation 1 : Synthese de conformite par departement
# Cette table permet d'analyser les tendances regionales
if "departements" in gold_tables and gold_tables["departements"] is not None:
    print("\nCreation : gold_conformite_agregee_departement")
    try:
        df_dept = gold_tables["departements"]
        
        # Groupement par departement avec calculs agreges
        conformite_dept = df_dept.groupBy("code_departement").agg(
            avg("pollution_score").alias("score_moyen"),
            sum("alerts").alias("total_alertes"),
            count("id_prelevement").alias("nb_prelevements")
        ).orderBy(col("score_moyen").desc())
        
        # Persistence en table Delta
        conformite_dept.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("gold_conformite_agregee_departement")
        
        print("   OK - Table creee")
        display(conformite_dept.limit(5))
        
    except Exception as e:
        print("   Attention - Agregation departement impossible : " + str(e))

# Agregation 2 : Synthese des parametres critiques
# Cette table centralise les indicateurs des parametres hors normes
if "critical" in gold_tables and gold_tables["critical"] is not None:
    print("\nCreation : gold_synthese_parametres_critiques")
    try:
        df_crit = gold_tables["critical"]
        
        # Selection et filtrage des donnees pertinentes
        synthese_critique = df_crit.select(
            col("parameter"),
            col("count").alias("nb_mesures"),
            col("mean").alias("valeur_moyenne"),
            col("median").alias("valeur_mediane"),
            col("percent_exceeding").alias("pct_hors_seuil")
        ).filter(col("nb_mesures") > 0)
        
        # Persistence en table Delta
        synthese_critique.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("gold_synthese_parametres_critiques")
        
        print("   OK - Table creee")
        display(synthese_critique)
        
    except Exception as e:
        print("   Attention - Agregation parametres critiques impossible : " + str(e))


